In [4]:
# #region agent log
import json, time, importlib.util
_log = "/home/bento/analytics-sql/consumers/srividyasekar/.cursor/debug-7a7820.log"
def _dbg(hid, msg, data=None):
    with open(_log, "a") as f:
        f.write(json.dumps({"sessionId":"7a7820","runId":"post-fix","hypothesisId":hid,"location":"cx_taxonomy.ipynb:install","message":msg,"data":data or {},"timestamp":int(time.time()*1000)}) + "\n")
_dbg("H1", "install_cell_start", {"uses_pip_magic": True})
# #endregion

# Use %pip so installs go to this notebook kernel's environment
%pip install -q pandas instaquery numpy matplotlib openai snowflake-connector-python

# #region agent log
for _pkg in ["pandas", "instaquery", "numpy", "matplotlib", "openai", "snowflake"]:
    _found = importlib.util.find_spec(_pkg.split(".")[0]) is not None
    _dbg("H4", "import_check", {"package": _pkg, "found": _found})
# #endregion


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
virtualenv 20.13.0+ds requires platformdirs<3,>=2, but you have platformdirs 4.10.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import libraries
import pandas as pd
import instaquery as iq
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import json
# Import librare
import instaquery as iq
import datetime
import numpy as np
import os
import logging 
snowflake_logger = logging.getLogger('snowflake.connector')
snowflake_logger.setLevel(logging.WARNING)
# Import regex if not already imported
import re

ModuleNotFoundError: No module named 'pandas'

# Pre-Processing: Summarize contact content to one-sentence

In [2]:
sql_query = """
WITH retail_agents AS (

SELECT
    ca.name
    , ca.id 
    -- , ca.UJET_AGENT_ID
    , ca.email
    , ra.team 
    , ra.supervisor
FROM etl.support_tool_users AS ca
INNER JOIN
    (
    SELECT
        agents.*,
        LOWER(agents.work_email) as email
        , SUBSTRING(agents.BUSINESS_TITLE, 21, 12) as team
        , manager.FIRST_NAME || ' ' ||manager.LAST_NAME as supervisor
        -- , agents.FIRST_NAME  || ' ' ||agents.LAST_NAME as specialist_name
        -- , manager2.FIRST_NAME || ' ' ||manager2.LAST_NAME as supervisor
    FROM dwh.workday_team_hierarchy as agents
    LEFT JOIN dwh.workday_team_hierarchy as manager
        ON agents.manager_id = manager.employee_id
    LEFT JOIN dwh.workday_team_hierarchy as manager2
        ON manager.manager_id = manager2.employee_id
    
    WHERE true
        AND agents.cost_center = 'Customer Experience Support'
        AND agents.BUSINESS_TITLE IN
        (
         'Customer Experience Retail Voice Specialist',
         'Customer Experience Retail Voice Specialists',
         'Customer Experience Retail Email Specialist',
         'Customer Experience Retail Voice Specialist II',
         'Customer Experience Retail Email Specialist II'
        )
    ) AS ra ON ca.email = ra.email

-- WHERE team = 'Retail Email'
)

,email_base as 
(SELECT 
    FSC.primary_contact_id,
    CSE.SUBJECT,
    FSC.subcase_type,
    FSC.CONTACT_CHANNEL,
    CSE.TEXT_BODY,
    CONTACT_CREATED_AT_UTC,
    CSE.CREATED_DATE,
    max(case when ca.id is not null then 1 else 0 end) as is_retail_agent,
    rank() over (partition by primary_contact_id order by CSE.CREATED_DATE asc) as message_rank,
    rank() over (partition by primary_contact_id order by CSE.CREATED_DATE desc) as message_rank_desc

FROM cx_support.salesforce_eclipse.emailmessage CSE    
JOIN INSTADATA.ETL.FACT_SUPPORT_CONTACTS FSC
ON cse.related_to_id = fsc.ticket_id::varchar
AND FSC.contact_channel = 'email'
INNER JOIN etl.fact_support_touches AS fst
ON fst.contact_id = fsc.id
LEFT JOIN retail_agents AS ca
ON ca.id = fst.agent_id
WHERE CONTACT_CREATED_AT_UTC::DATE >= '2025-05-01'
AND CONTACT_CREATED_AT_UTC::DATE < '2025-11-01'
AND FSC.USER_CHANNEL = 'retailer'
AND FSC.TICKET_ID_SOURCE = 'salesforce'
group by 1,2,3,4,5,6,7)

SELECT 
    primary_contact_id,
    CONTACT_CHANNEL,
    min(CREATED_DATE) as transcript_created_date_at_utc,
    max(is_retail_agent) as is_retail_agent,
    MAX(CASE WHEN message_rank = 1 THEN subject END) AS subject,
    LISTAGG(
        'Message ' || message_rank || ' (' || contact_created_at_utc || '):\n' || text_body, 
        '\n\n=== NEXT MESSAGE ===\n\n'
    ) WITHIN GROUP (ORDER BY CREATED_DATE) AS transcript
    --COUNT(*) as message_count
FROM email_base 
GROUP BY 1,2
--where (message_rank = 1 or message_rank_desc = 1)

UNION ALL

SELECT
    FSC.primary_contact_id,
    FSC.CONTACT_CHANNEL,
    min(CT.CREATED_AT_UTC) as transcript_created_date_at_utc,
    max(case when ca.id is not null then 1 else 0 end) as is_retail_agent,
    null as subject,
    max(CT.transcript) as transcript

FROM instadata.etl_eclipse.support_call_transcripts CT
JOIN INSTADATA.ETL.FACT_SUPPORT_CONTACTS FSC
ON FSC.primary_contact_id::varchar = CT.AUDIO_ID::varchar
AND CT.audio_source = replace(split(fsc.primary_contact_id_source, '_')[0],'"','')
INNER JOIN etl.fact_support_touches AS fst
ON fst.contact_id = fsc.id
LEFT JOIN retail_agents AS ca
ON ca.id = fst.agent_id
WHERE CONTACT_CREATED_AT_UTC::DATE >= '2025-05-01'
AND CONTACT_CREATED_AT_UTC::DATE < '2025-11-01'
AND FSC.USER_CHANNEL = 'retailer'
AND FSC.TICKET_ID_SOURCE = 'salesforce'
group by 1,2;
"""

In [3]:
results_df = iq.query(sql_query)

In [4]:
# @title Connecting with OpenAI
from openai import OpenAI

## Enter the name of the source you have created via the website or Bento UI
SOURCE = 'ashleyhan-personal'


client = OpenAI(
    # AI Gateway does not need or want an API key, passed unused because its required here
    api_key="unused",
    base_url=f"https://aigateway.instacart.tools/proxy/{SOURCE}/openai/v1"
)

In [13]:
df = pd.read_csv('/Users/ashleyhan/Documents/data/cx_retailer_sampling.csv')
df.head()

,CASE_LINK,CONTACT_CHANNEL,SUBJECT,TRANSCRIPT_MASKED,TRANSCRIPT_CLEANED_MASKED,SUMMARY_LLM,SUBCASE_TYPE_CX,RETAILER_FOD,PRODUCT_FOD,RETAILER_LLM,PRODUCT_LLM,ORDER_ID_FOD,ORDER_ID_LLM,CUSTOMER_ID_LLM,IS_RETAILER_AGENT_CX
0,https://instacartcustomerexperience.lightning....,email,Wegmans Customer Care # 4028611 From Kendra ...,"Message 1 (2025-10-20 14:55:09.000 Z):\nHello,...","Hello,\n\nBelow is a customer communication th...",Customer reported a shopper ignored both writt...,retailer :: order_issue :: shopper_issue,Wegmans,API,Wegmans,NaN,1.828335e+16,20902996811211325445,104268305,1
1,https://instacartcustomerexperience.lightning....,phone,NaN,[15:28:11 agent]: thank you for calling instat...,[15:28:11 agent]: thank you for calling instat...,Retailer reported an issue with completing a s...,shopper :: fulfillment :: checkout_issue,Hy-Vee,API,NaN,NaN,1.830484e+16,39387462,NaN,1
2,https://instacartcustomerexperience.lightning....,phone,NaN,[23:18:12 agent]: thank you for contacting ans...,[23:18:12 agent]: thank you for contacting ans...,Retailer inquired about a late order which was...,consumer :: order_issue :: order_status,Kroger,API,Kroger,LMD,1.821188e+16,NaN,NaN,1
3,https://instacartcustomerexperience.lightning....,phone,NaN,[17:28:29 agent]: thank you for calling suppor...,[17:28:29 agent]: thank you for calling suppor...,Retailer experienced an issue with a high orde...,consumer :: payment_method :: declined,Aldi,API,NaN,NaN,1.792880e+16,NaN,NaN,1
4,https://instacartcustomerexperience.lightning....,phone,NaN,[19:43:16 agent]: thank you for calling instal...,[19:43:16 agent]: thank you for calling instal...,Customer's order was delayed due to high order...,retailer :: order_issue :: order_status,Kroger,API,Kroger,LMD,1.830548e+16,NaN,NaN,1


## summarization 

In [35]:
system_prompt = """You are a customer experience analyst specializing in contact reason classification. 
Your task is to read customer service transcripts and create a single, concise sentence that captures 
the PRIMARY reason the audience contacted support.

Guidelines:
- Focus on the customer's CORE ISSUE or REQUEST, not secondary concerns
- Use neutral, descriptive language (avoid subjective terms)
- Capture WHAT happened and WHAT the customer wanted
- Start with an action verb when possible (e.g., "Customer/Shopper/Retailer reported...", "Customer/Shopper/Retailer requested...", "Customer/Shopper/Retailer inquired...")
- Include key details: order/delivery issues, payment problems, product concerns, account issues, etc.
- Keep it under 20 words
- Do NOT include:
  * Agent names or internal processes
  * Pleasantries or conversational filler
  * Resolution details (focus on the initial problem)
  * Multiple issues (choose the primary one)

Examples:
- "Customer reported order was never delivered and requested a refund"
- "Customer inquired about missing items from their completed order"
- "Customer requested cancellation of recurring subscription charges"
- "Customer reported delivery driver left groceries at wrong address"
- "Customer escalated complaint about damaged produce received in order"
"""

user_prompt_template = """Transcript:
{TRANSCRIPT}

Summarize the primary contact reason in ONE sentence (under 25 words):"""

In [33]:
# Check what columns you actually have
print("Column names in results_df:")
print(results_df.columns.tolist())
print("\nFirst row preview:")
print(results_df.head(1))

Column names in results_df:
['primary_contact_id', 'contact_channel', 'transcript_created_date_at_utc', 'is_retail_agent', 'subject', 'transcript']

First row preview:
   primary_contact_id contact_channel transcript_created_date_at_utc  \
0  500Uc00000YA60cIAD           email            2025-05-05 21:31:37   

   is_retail_agent                                            subject  \
0                1  Wegmans Customer Care # 3841428   From Erica C...   

                                          transcript  
0  Message 1 (2025-05-05 21:31:37.000 Z):\nHello,...  


In [37]:
# DIAGNOSTIC: Delete checkpoint and test ONE row manually
import os

# 1. Delete the bad checkpoint
checkpoint_file = 'cx_summary_checkpoint.csv'
if os.path.exists(checkpoint_file):
    os.remove(checkpoint_file)
    print(f"✅ Deleted bad checkpoint: {checkpoint_file}")

# 2. Test ONE row manually to see the real error
print("\n=== MANUAL TEST ON ONE ROW ===")
test_row = results_df.iloc[0]

print(f"Transcript length: {len(str(test_row['transcript']))}")
print(f"Subject: {test_row['subject']}")
print(f"Channel: {test_row['contact_channel']}")

try:
    # Test the API call
    result = summarize_transcript(
        transcript=test_row['transcript'],
        subject=test_row['subject'],
        contact_channel=test_row['contact_channel']
    )
    print(f"\n✅ SUCCESS! Result: {result}")
    
except Exception as e:
    print(f"\n❌ ERROR FOUND!")
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {str(e)}")
    import traceback
    print(f"\nFull traceback:")
    traceback.print_exc()

✅ Deleted bad checkpoint: cx_summary_checkpoint.csv

=== MANUAL TEST ON ONE ROW ===
Transcript length: 4379
Subject: Wegmans Customer Care # 3841428   From Erica Customer Care Center
Channel: email

✅ SUCCESS! Result: Customer reported groceries smelled like marijuana and were poorly packed, requesting a refund of $72.57.


In [38]:
# ============================================
# PROCESS ALL RECORDS WITH PARALLEL PROCESSING
# ============================================
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def process_single_row(idx, row):
    """Process a single row with error handling"""
    try:
        # Handle potential missing data
        transcript = row.get('transcript')
        if not transcript or pd.isna(transcript):
            return idx, None, "No transcript available"
        
        subject = row.get('subject')
        channel = row.get('contact_channel')
        
        summary = summarize_transcript(transcript, subject, channel)
        return idx, summary, None
    except Exception as e:
        return idx, None, f"{type(e).__name__}: {str(e)}"


def summarize_parallel(df, max_workers=10, checkpoint_every=100, checkpoint_file='cx_summary_checkpoint.csv'):
    """
    Process transcripts in parallel with checkpointing
    
    Args:
        df: Input dataframe
        max_workers: Number of parallel threads (5-15 recommended)
        checkpoint_every: Save progress every N completions
        checkpoint_file: Where to save progress
    """
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['summary'].notna().sum()
        print(f"✅ Already completed: {already_done} rows")
    else:
        df = df.copy()
        df['summary'] = None
        df['error'] = None
    
    # Get rows that still need processing
    to_process = df[df['summary'].isna()]
    
    if len(to_process) == 0:
        print("🎉 All rows already processed!")
        return df
    
    print(f"🚀 Processing {len(to_process)} rows with {max_workers} parallel workers...")
    print(f"💾 Checkpointing every {checkpoint_every} completions to {checkpoint_file}")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / (max_workers * 1) / 60:.1f} minutes\n")
    
    completed_count = 0
    start_time = time.time()
    
    # Submit all tasks to thread pool
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Create futures
        future_to_idx = {
            executor.submit(process_single_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        # Process as they complete
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="Summarizing"):
            idx, summary, error = future.result()
            
            if summary:
                df.at[idx, 'summary'] = summary
            if error:
                df.at[idx, 'error'] = error
                # Only print first few errors to avoid spam
                if completed_count < 5:
                    print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint periodically
            if completed_count % checkpoint_every == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint saved. Progress: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%)")
                print(f"   Rate: {rate:.1f} rows/sec, ETA: {eta_seconds/60:.1f} min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    success_count = df['summary'].notna().sum()
    error_count = df['error'].notna().sum()
    
    print(f"\n✅ COMPLETE! Processed {len(to_process)} rows in {elapsed/60:.1f} minutes")
    print(f"📊 Success: {success_count} ({success_count/len(df)*100:.1f}%)")
    print(f"⚠️  Errors: {error_count} ({error_count/len(df)*100:.1f}%)")
    print(f"⚡ Average rate: {len(to_process)/elapsed:.1f} rows/sec")
    
    return df


# ============================================
# RUN THE PROCESSING
# ============================================
print(f"📋 Total rows in results_df: {len(results_df):,}")
print(f"🎯 Starting parallel processing...\n")

# Process with 10 parallel workers (adjust if you hit rate limits)
results_with_summary = summarize_parallel(
    results_df, 
    max_workers=10,  # Reduce to 5 if you see rate limit errors
    checkpoint_every=100
)

print("\n" + "="*60)
print("SAMPLE RESULTS:")
print("="*60)
print(results_with_summary[['subject', 'summary', 'error']].head(20))


📋 Total rows in results_df: 36,843
🎯 Starting parallel processing...

🚀 Processing 36843 rows with 10 parallel workers...
💾 Checkpointing every 100 completions to cx_summary_checkpoint.csv
⏱️  Estimated time: 30.7 - 61.4 minutes



Summarizing:   0%|          | 101/36843 [00:08<2:51:58,  3.56it/s]


💾 Checkpoint saved. Progress: 100/36843 (0.3%)
   Rate: 9.5 rows/sec, ETA: 64.1 min


Summarizing:   1%|          | 200/36843 [00:14<1:59:53,  5.09it/s]


💾 Checkpoint saved. Progress: 200/36843 (0.5%)
   Rate: 11.9 rows/sec, ETA: 51.2 min


...


💾 Checkpoint saved. Progress: 36700/36843 (99.6%)
   Rate: 12.8 rows/sec, ETA: 0.2 min


Summarizing: 100%|█████████▉| 36821/36843 [47:45<00:01, 16.86it/s]


💾 Checkpoint saved. Progress: 36800/36843 (99.9%)
   Rate: 12.8 rows/sec, ETA: 0.1 min


Summarizing: 100%|██████████| 36843/36843 [47:46<00:00, 12.85it/s]



✅ COMPLETE! Processed 36843 rows in 47.8 minutes
📊 Success: 36842 (100.0%)
⚠️  Errors: 1 (0.0%)
⚡ Average rate: 12.8 rows/sec

SAMPLE RESULTS:
                                              subject  \
0   Wegmans Customer Care # 3841428   From Erica C...   
1   Wegmans Customer Care # 3859502   From Jordan ...   
2   ERS-448 RE: Request to Add Email Addresses for...   
3                      please add to name/description   
4   [EXTERNAL] Store Hours Change Tracker: Store#5...   
5   [EXTERNAL] Store Hours Change Tracker: Store#2...   
6                               Merchant Fee Question   
7                 Standard - Case Ref # 5077611 - FYI   
8   Wegmans Customer Care # 3873163   From Jamie C...   
9   PSO-53486 Instacart Enterprise Technical Suppo...   
10  FW: Omnichannel Feedback Form - 1677  Counsel ...   
11  FW: Omnichannel Feedback Form - 1775  Counsel ...   
12  [EXTERNAL] Store Hours Change Tracker: Store#2...   
13                                          Invoicing   
1

In [ ]:
# ============================================
# SAVE FINAL RESULTS
# ============================================

# Save to CSV for further analysis
output_file = '/Users/ashleyhan/Documents/data/cx_retailer_transcripts_summarized.csv'
results_with_summary.to_csv(output_file, index=False)
print(f"💾 Saved results to: {output_file}")


iq.upload(results_with_summary, "SANDBOX_DB_PII.ASHLEYHAN.CX_RETAILER_CONTACTS_SUMMARY", if_exists="replace")

# Show statistics
print("\n" + "="*60)
print("FINAL STATISTICS:")
print("="*60)
print(f"Total records: {len(results_with_summary):,}")
print(f"Successfully summarized: {results_with_summary['summary'].notna().sum():,}")
print(f"Errors: {results_with_summary['error'].notna().sum():,}")
print(f"\nSuccess rate: {results_with_summary['summary'].notna().sum() / len(results_with_summary) * 100:.2f}%")

# Show error breakdown if any
if results_with_summary['error'].notna().sum() > 0:
    print("\n" + "="*60)
    print("ERROR BREAKDOWN:")
    print("="*60)
    print(results_with_summary['error'].value_counts().head(10))

# Preview successful summaries
print("\n" + "="*60)
print("SAMPLE SUCCESSFUL SUMMARIES:")
print("="*60)
successful = results_with_summary[results_with_summary['summary'].notna()].head(20)
for idx, row in successful.iterrows():
    print(f"\n{idx}. Channel: {row['contact_channel']}")
    if pd.notna(row['subject']):
        print(f"   Subject: {row['subject'][:80]}...")
    print(f"   Summary: {row['summary']}")


💾 Saved results to: /Users/ashleyhan/Documents/data/cx_retailer_transcripts_summarized.csv

FINAL STATISTICS:
Total records: 36,843
Successfully summarized: 36,842
Errors: 1

Success rate: 100.00%

ERROR BREAKDOWN:
error
No transcript available    1
Name: count, dtype: int64

SAMPLE SUCCESSFUL SUMMARIES:

0. Channel: email
   Subject: Wegmans Customer Care # 3841428   From Erica Customer Care Center...
   Summary: Customer reported groceries smelled like marijuana and were poorly packed, requesting a refund of $72.57.

1. Channel: email
   Subject: Wegmans Customer Care # 3859502   From Jordan Customer Care Center...
   Summary: Customer reported delivery driver did not follow instructions to leave order at door and requested a callback.

2. Channel: email
   Subject: ERS-448 RE: Request to Add Email Addresses for Instacart Error Notifications - I...
   Summary: Customer requested to add email addresses for receiving Instacart error notifications.

3. Channel: email
   Subject: please 

# 🏗️ TAXONOMY BUILDING PIPELINE

This pipeline will:
1. **Step 1**: Load summarized data and explore patterns
2. **Step 2**: GPT-assisted L1 category discovery (3-6 top-level categories)
3. **Step 3**: Validate and refine L1 categories
4. **Step 4**: GPT-assisted L2 subcategory discovery for each L1
5. **Step 5**: Classify all 36k summaries into L1 categories
6. **Step 6**: Classify all summaries into L2 categories
7. **Step 7**: Review, validate, and export final taxonomy


# 📚 Quick Reference: How to Use This Pipeline

## Pipeline Overview
This notebook creates a hierarchical taxonomy (L1 → L2) for CX contact reasons using GPT-4 with **stratified sampling by channel** and a **holdout evaluation set** for unbiased validation.

## 🎯 Key Features
- **Stratified Sampling**: 70% phone/voice, 30% email in all sampling stages
- **Holdout Evaluation**: 500 clean samples never seen during discovery
- **Fast Validation**: Classify only 500 eval samples (~5-10 min) instead of all 36k
- **Checkpointing**: Auto-resume if interrupted

## Execution Order
Run cells in this order:

### Discovery Phase (Build Taxonomy)
1. **Cell 13**: Load summarized data → `df_valid` (36k summaries)
2. **Cell 14**: Discover L1 categories (700 phone + 300 email) → `l1_result`, `l1_used_indices`
3. **Cell 15**: Discover L2 subcategories (350 phone + 150 email per L1) → `l2_complete_taxonomy`, `l2_used_indices`

### Evaluation Phase (Test Taxonomy)
4. **Cell 16**: Import regex module (if needed)
5. **Cell 17**: Create holdout evaluation set (350 phone + 150 email, excluded from discovery) → `df_eval` (500 samples)
6. ⚠️ **BEFORE CLASSIFICATION**: Delete old checkpoints!
   ```python
   !rm -f classification_l1_checkpoint.csv classification_l2_checkpoint.csv
   !rm -f classification_l1_eval_checkpoint.csv classification_l2_eval_checkpoint.csv
   ```
7. **Cell 18/19**: Classify eval set into L1 (~2-5 min) → `df_with_l1`
8. **Cell 19/20**: Classify eval set into L2 (~2-5 min) → `df_with_l2`
9. **Cell 20/21**: Validate classification quality
10. **Cell 21/22**: Export final results

## 📊 Sample Sizes Summary

| Stage | Phone Samples | Email Samples | Total | Purpose |
|-------|--------------|---------------|-------|---------|
| **L1 Discovery** | 700 | 300 | 1,000 | Propose L1 categories |
| **L2 Discovery** | 350 per L1 | 150 per L1 | ~2,500 total | Propose L2 subcategories |
| **Holdout Eval** | 350 | 150 | 500 | Test taxonomy (zero overlap) |

## Output Files
- `taxonomy_l1_proposal.json` - Initial L1 category proposals
- `taxonomy_complete_proposal.json` - Complete L1+L2 taxonomy structure
- `classification_l1_eval_checkpoint.csv` - L1 classification results (eval set)
- `classification_l2_eval_checkpoint.csv` - L2 classification results (eval set)
- `cx_transcripts_with_taxonomy.csv` - Final classified data (if you run on full dataset)
- `cx_taxonomy_final.json` - Taxonomy definition for production use
- `cx_taxonomy_report.json` - Distribution statistics and summary

## ⚠️ Before Running Classification
**IMPORTANT**: Delete old checkpoint files to ensure fresh evaluation:
```python
!rm -f classification_*_checkpoint.csv classification_*_eval_checkpoint.csv
```
Or run the checkpoint cleanup cell before classification.

## Key Parameters to Adjust

### Sampling Strategy
- **L1 phone/email split**: Change `phone_samples=700, email_samples=300` in Cell 14
- **L2 phone/email split**: Change `phone_samples=350, email_samples=150` in Cell 15
- **Eval set size**: Adjust sampling in Cell 17 (currently 350 phone, 150 email)

### Taxonomy Size
- **L1 categories**: Change `num_l1_categories=6` in Cell 14
- **L2 subcategories**: Change `num_l2=5` in Cell 15

### Performance
- **Parallel workers**: Change `max_workers=10` if hitting rate limits (reduce to 5)
- **Checkpoint frequency**: Adjust `checkpoint_every=100` in classification calls

## Estimated Runtime

### Discovery Phase
- L1 Discovery: ~2 minutes (1,000 samples)
- L2 Discovery: ~5-10 minutes (~2,500 samples across all L1s)

### Evaluation Phase (500 samples)
- L1 Classification: ~2-5 minutes
- L2 Classification: ~2-5 minutes
- **Total eval time: ~10-20 minutes**

### Full Dataset Classification (if needed)
- L1 Classification: 30-60 minutes (36k records)
- L2 Classification: 30-60 minutes (36k records)

## 🔄 Iterating on Taxonomy

### Option 1: Refine and Re-Evaluate
1. Review validation output
2. Manually edit `l2_complete_taxonomy` JSON structure
3. Delete eval checkpoint files: `!rm classification_*_eval_checkpoint.csv`
4. Re-run classification cells on eval set

### Option 2: Rebuild from Scratch
1. Adjust parameters in Cells 14-15
2. Re-run discovery (Cells 14-15)
3. Create new eval set (Cell 17)
4. Delete all checkpoints
5. Re-run classification

## 🎯 Why Holdout Evaluation?
The 500-sample eval set:
- ✅ **Never seen during taxonomy discovery** (L1 or L2)
- ✅ **Unbiased validation** of taxonomy quality
- ✅ **Fast iteration** (~10 min vs 2 hours)
- ✅ **Representative** (70/30 phone/email split)
- ✅ **Clean for metrics** (precision, recall, confusion matrix)

## 📈 Scaling to Full Dataset
Once taxonomy is validated on eval set:
1. Modify classification cell to use `df_valid` instead of `df_eval`
2. Change checkpoint filenames (e.g., `classification_l1_full.csv`)
3. Increase `checkpoint_every` to 500 or 1000
4. Budget 1-2 hours for full classification


In [40]:
# ============================================
# STEP 1: LOAD SUMMARIZED DATA
# ============================================

# Load the summarized transcripts
summary_file = '/Users/ashleyhan/Documents/data/cx_retailer_transcripts_summarized.csv'
df_summaries = pd.read_csv(summary_file)

print(f"📊 Loaded {len(df_summaries):,} summarized transcripts")
print(f"✅ Valid summaries: {df_summaries['summary'].notna().sum():,}")
print(f"⚠️  Missing/Error summaries: {df_summaries['summary'].isna().sum():,}")

# Filter to only valid summaries for taxonomy building
df_valid = df_summaries[df_summaries['summary'].notna()].copy()
df_valid = df_valid[df_valid['summary'] != 'ERROR'].copy()

print(f"\n🎯 Working with {len(df_valid):,} valid summaries for taxonomy")

# Show sample
print("\n📋 Sample summaries:")
for i, row in df_valid.head(10).iterrows():
    print(f"{i+1}. {row['summary']}")


📊 Loaded 36,843 summarized transcripts
✅ Valid summaries: 36,842
⚠️  Missing/Error summaries: 1

🎯 Working with 36,842 valid summaries for taxonomy

📋 Sample summaries:
1. Customer reported groceries smelled like marijuana and were poorly packed, requesting a refund of $72.57.
2. Customer reported delivery driver did not follow instructions to leave order at door and requested a callback.
3. Customer requested to add email addresses for receiving Instacart error notifications.
4. Customer requested to add "bulk frozen" to the name and description of an item.
5. Retailer reported a change in store hours for location #520 due to limited staffing challenges.
6. Retailer reported early store closure due to customer threat and communicated updated special hours for safety reasons.
7. Customer inquired about a merchant fee charge on their invoice despite a recent markup increase to 12%.
8. Customer reported an Instacart delivery driver ran over her child's bicycle and requested assistance wi

In [72]:
# ============================================
# STEP 2: GPT-ASSISTED L1 CATEGORY DISCOVERY
# ============================================

def discover_l1_categories(summaries, phone_samples=700, email_samples=300, num_l1_categories=4):
    """
    Use GPT to analyze summaries and propose L1 categories
    Uses stratified sampling by channel to ensure both phone and email are well-represented
    """
    # Stratified sampling by contact_channel
    # Get phone/voice contacts
    phone_data = summaries[summaries['contact_channel'].str.lower().isin(['phone', 'voice'])].copy()
    # Get email contacts
    email_data = summaries[summaries['contact_channel'].str.lower() == 'email'].copy()
    
    # Sample from each channel
    phone_sample_size = min(phone_samples, len(phone_data))
    email_sample_size = min(email_samples, len(email_data))
    
    phone_sample = phone_data.sample(n=phone_sample_size, random_state=42) if len(phone_data) > 0 else pd.DataFrame()
    email_sample = email_data.sample(n=email_sample_size, random_state=42) if len(email_data) > 0 else pd.DataFrame()
    
    # Combine samples
    combined_sample = pd.concat([phone_sample, email_sample], ignore_index=True)
    sample_summaries = combined_sample['summary'].tolist()
    used_indices = combined_sample.index.tolist()  # Track which rows were used
    
    print(f"🔍 Stratified sampling by channel:")
    print(f"   📞 Phone/Voice: {len(phone_sample)} samples (from {len(phone_data):,} total)")
    print(f"   📧 Email: {len(email_sample)} samples (from {len(email_data):,} total)")
    print(f"   📊 Total samples: {len(sample_summaries)}")
    
    # Format samples
    samples_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sample_summaries))
    
    # GPT prompt for L1 discovery
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.3,
        messages=[
            {
                'role': 'system',
                'content': """You are an expert CX taxonomist building a contact reason classification system.
Your goal is to identify the top-level (L1) categories that represent the main reasons customers/retailers contact support.

Guidelines:
- Create 4-7 L1 categories that are mutually exclusive and collectively exhaustive
- Focus on the REASON for contact, not channel or outcome
- Use business-friendly names (e.g., "Account Management" not "Account Stuff")
- Each L1 should represent ~10-30% of contacts (avoid tiny or huge categories)
- Consider the voice of customer: what would they say they're contacting about?

Output format: JSON with L1 categories, descriptions, keywords, and estimated %"""
            },
            {
                'role': 'user',
                'content': f"""Analyze these {len(sample_summaries)} customer contact summaries and identify {num_l1_categories} top-level (L1) categories.

SUMMARIES:
{samples_text}

TASK:
1. Identify {num_l1_categories} L1 categories
2. For each L1, provide:
   - Clear name
   - Business description (1 sentence)
   - Top 20 keywords/phrases
   - Estimated % of total contacts
   - 5 example IDs from the list above

Output JSON in this exact schema:
{{
  "L1_categories": [
    {{
      "id": "L1_01",
      "name": "Category Name",
      "description": "What this category covers",
      "keywords": ["keyword1", "keyword2", "keyword3", "keyword4", "keyword5", "keyword6", "keyword7", "keyword8", "keyword9", "keyword10", "keyword11", "keyword12", "keyword13", "keyword14", "keyword15", "keyword16", "keyword17", "keyword18", "keyword19", "keyword20"],
      "estimated_pct": 25,
      "example_ids": [1, 15, 203, 456, 789]
    }}
  ],
  "analysis_notes": "Any observations about the data"
}}"""
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result, sample_summaries, used_indices


# Run L1 discovery
print("🚀 Starting L1 category discovery...")
l1_result, sample_summaries, l1_used_indices = discover_l1_categories(df_valid, phone_samples=700, email_samples=300, num_l1_categories=6)

print(f"\n📝 Tracking: {len(l1_used_indices)} indices used in L1 discovery")

# Display results
print("\n" + "="*80)
print("📊 PROPOSED L1 CATEGORIES")
print("="*80)

for cat in l1_result['L1_categories']:
    print(f"\n{cat['id']}: {cat['name']} (~{cat['estimated_pct']}%)")
    print(f"   Description: {cat['description']}")
    print(f"   Keywords: {', '.join(cat['keywords'][:5])}...")
    print(f"   Examples:")
    for ex_id in cat['example_ids'][:3]:
        if ex_id <= len(sample_summaries):
            print(f"      - {sample_summaries[ex_id-1]}")

if 'analysis_notes' in l1_result:
    print(f"\n📝 Analysis Notes: {l1_result['analysis_notes']}")

# Save L1 proposal
with open('taxonomy_l1_proposal.json', 'w') as f:
    json.dump(l1_result, f, indent=2)
print("\n💾 Saved L1 proposal to: taxonomy_l1_proposal.json")


🚀 Starting L1 category discovery...
🔍 Stratified sampling by channel:
   📞 Phone/Voice: 700 samples (from 32,625 total)
   📧 Email: 300 samples (from 4,217 total)
   📊 Total samples: 1000

📝 Tracking: 1000 indices used in L1 discovery

📊 PROPOSED L1 CATEGORIES

L1_01: Order Status and Delivery Issues (~30%)
   Description: Covers inquiries and complaints about order status, delays, missing deliveries, or incorrect delivery locations.
   Keywords: order status, delivery issue, delayed order, missing delivery, wrong address...
   Examples:
      - Retailer reported a customer did not receive their order and requested confirmation of delivery status.
      - Retailer reported a customer's order missing and requested GPS verification of the delivery location.
      - Retailer inquired about a customer's order that was not delivered on the scheduled date.

L1_02: Order Cancellations and Modifications (~25%)
   Description: Includes requests to cancel or modify orders due to customer or reta

In [78]:
# ============================================
# STEP 3: GPT-ASSISTED L2 SUBCATEGORY DISCOVERY
# ============================================

def discover_l2_categories(l1_category, summaries, phone_samples=350, email_samples=150, num_l2=5):
    """
    For a given L1 category, use GPT to propose L2 subcategories
    Uses stratified sampling by channel (70% phone, 30% email by default)
    With improved fallback: blends keyword matches with random samples when insufficient matches
    """
    # Filter summaries that likely belong to this L1 based on keywords
    keywords = l1_category['keywords']
    keyword_pattern = '|'.join([re.escape(k.lower()) for k in keywords[:10]])
    
    # Find summaries matching this L1's keywords
    matching = summaries[summaries['summary'].str.lower().str.contains(keyword_pattern, na=False, regex=True)]
    
    print(f"🔍 Discovering L2 subcategories for: {l1_category['name']}")
    print(f"   🔎 Keyword matches found: {len(matching)}")
    
    # Split matches by channel
    phone_matches = matching[matching['contact_channel'].str.lower().isin(['phone', 'voice'])]
    email_matches = matching[matching['contact_channel'].str.lower() == 'email']
    
    print(f"      Phone matches: {len(phone_matches)}, Email matches: {len(email_matches)}")
    
    # Check if we have enough matches per channel
    if len(phone_matches) < phone_samples or len(email_matches) < email_samples:
        print(f"   ⚠️  Insufficient keyword matches")
        print(f"   💡 Blending keyword matches with random samples...")
        
        # Use all keyword matches
        phone_sample = phone_matches.copy()
        email_sample = email_matches.copy()
        
        # Fill remaining with random samples (excluding keyword matches)
        if len(phone_sample) < phone_samples:
            phone_pool = summaries[
                (summaries['contact_channel'].str.lower().isin(['phone', 'voice'])) &
                (~summaries.index.isin(phone_matches.index))
            ]
            additional_needed = phone_samples - len(phone_sample)
            if len(phone_pool) > 0:
                additional_phone = phone_pool.sample(
                    n=min(additional_needed, len(phone_pool)),
                    random_state=42
                )
                phone_sample = pd.concat([phone_sample, additional_phone])
        
        if len(email_sample) < email_samples:
            email_pool = summaries[
                (summaries['contact_channel'].str.lower() == 'email') &
                (~summaries.index.isin(email_matches.index))
            ]
            additional_needed = email_samples - len(email_sample)
            if len(email_pool) > 0:
                additional_email = email_pool.sample(
                    n=min(additional_needed, len(email_pool)),
                    random_state=42
                )
                email_sample = pd.concat([email_sample, additional_email])
        
        # Track how many from keywords vs random
        phone_from_keywords = len([i for i in phone_sample.index if i in phone_matches.index])
        email_from_keywords = len([i for i in email_sample.index if i in email_matches.index])
        
        print(f"   📞 Phone: {len(phone_sample)} total ({phone_from_keywords} keywords + {len(phone_sample)-phone_from_keywords} random)")
        print(f"   📧 Email: {len(email_sample)} total ({email_from_keywords} keywords + {len(email_sample)-email_from_keywords} random)")
    else:
        # Enough matches, sample from keyword matches only
        phone_sample = phone_matches.sample(n=min(phone_samples, len(phone_matches)), random_state=42)
        email_sample = email_matches.sample(n=min(email_samples, len(email_matches)), random_state=42)
        
        print(f"   ✅ Sufficient keyword matches")
        print(f"   📞 Phone: {len(phone_sample)} samples (all from keywords)")
        print(f"   📧 Email: {len(email_sample)} samples (all from keywords)")
    
    # Combine samples
    combined_sample = pd.concat([phone_sample, email_sample])
    sample_summaries = combined_sample['summary'].tolist()
    used_indices = combined_sample.index.tolist()
    
    sample_text = "\n".join(f"{i+1}. {s}" for i, s in enumerate(sample_summaries))
    
    print(f"   📊 Total samples for L2 discovery: {len(sample_summaries)}")
    
    # Rest of the function stays the same (GPT call)
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.3,
        messages=[
            {
                'role': 'system',
                'content': """You are an expert CX taxonomist creating detailed subcategories (L2) within a top-level category (L1).

Guidelines:
- Create 3-7 L2 subcategories that are mutually exclusive within this L1
- L2 categories should be specific and actionable
- Each L2 should represent a distinct contact reason or issue type
- Use clear, business-friendly names
- Focus on what would help teams route and prioritize contacts"""
            },
            {
                'role': 'user',
                'content': f"""You are creating L2 subcategories for the L1 category: "{l1_category['name']}"

L1 Description: {l1_category['description']}

Here are {len(sample_summaries)} contact summaries that belong to this L1 category:

{sample_text}

TASK:
Create {num_l2} L2 subcategories that divide "{l1_category['name']}" into meaningful, specific groups.

Output JSON in this schema:
{{
  "L2_categories": [
    {{
      "id": "L2_01",
      "name": "Subcategory Name",
      "description": "What this subcategory covers",
      "keywords": ["keyword1", "keyword2", ...],
      "example_ids": [1, 5, 12]
    }}
  ]
}}"""
            }
        ],
        response_format={"type": "json_object"}
    )
    
    result = json.loads(response.choices[0].message.content)
    return result, matching['summary'].tolist(), used_indices


# Discover L2 for each L1
print("🚀 Starting L2 subcategory discovery for each L1...")
print("="*80)

l2_complete_taxonomy = {
    'L1_categories': []
}

l2_used_indices = []  # Track all indices used across all L2 discoveries

for l1_cat in l1_result['L1_categories']:
    l2_result, l2_samples, l2_indices = discover_l2_categories(l1_cat, df_valid, phone_samples=350, email_samples=150, num_l2=8)
    l2_used_indices.extend(l2_indices)  # Accumulate indices
    
    # Add L2 categories to the L1 category
    l1_with_l2 = l1_cat.copy()
    l1_with_l2['L2_categories'] = l2_result['L2_categories']
    
    # Update L2 IDs to include L1 prefix
    for i, l2_cat in enumerate(l1_with_l2['L2_categories']):
        l2_cat['id'] = f"{l1_cat['id']}_L2_{i+1:02d}"
    
    l2_complete_taxonomy['L1_categories'].append(l1_with_l2)
    
    # Display
    print(f"\n✅ {l1_cat['name']}:")
    for l2_cat in l1_with_l2['L2_categories']:
        print(f"   └─ {l2_cat['id']}: {l2_cat['name']}")
        print(f"      {l2_cat['description']}")
        print(f"      Keywords: {', '.join(l2_cat['keywords'][:3])}...")

# Save complete taxonomy
with open('taxonomy_complete_proposal.json', 'w') as f:
    json.dump(l2_complete_taxonomy, f, indent=2)

print("\n" + "="*80)
print("💾 Saved complete L1+L2 taxonomy to: taxonomy_complete_proposal.json")
print("="*80)

# Track all indices used in discovery
all_discovery_indices = set(l1_used_indices + l2_used_indices)
print(f"\n📝 Discovery dataset tracking:")
print(f"   L1 discovery: {len(l1_used_indices)} unique indices")
print(f"   L2 discovery: {len(set(l2_used_indices))} unique indices (across all L1s)")
print(f"   Combined unique: {len(all_discovery_indices)} indices used in discovery")
print(f"   Available for holdout: {len(df_valid) - len(all_discovery_indices):,} indices")


🚀 Starting L2 subcategory discovery for each L1...
🔍 Discovering L2 subcategories for: Order Status and Delivery Issues
   🔎 Keyword matches found: 2879
      Phone matches: 2815, Email matches: 64
   ⚠️  Insufficient keyword matches
   💡 Blending keyword matches with random samples...
   📞 Phone: 2815 total (2815 keywords + 0 random)
   📧 Email: 150 total (64 keywords + 86 random)
   📊 Total samples for L2 discovery: 2965

✅ Order Status and Delivery Issues:
   └─ L1_01_L2_01: Order Status Updates
      Covers inquiries about the current status of an order, including delays, lack of updates, or shopper assignment issues.
      Keywords: order status, status update, delayed order...
   └─ L1_01_L2_02: Delayed Deliveries
      Covers issues related to orders that are delayed beyond the expected delivery window, including requests for rescheduling or cancellation.
      Keywords: delayed delivery, reschedule, late order...
   └─ L1_01_L2_03: Missing Deliveries
      Covers reports of ord

In [86]:
# ============================================
# CREATE HOLDOUT EVALUATION SET
# ============================================

print("🎯 Creating holdout evaluation set...")
print("="*80)

# Get all indices NOT used in L1 or L2 discovery
holdout_pool = df_valid[~df_valid.index.isin(all_discovery_indices)].copy()

print(f"📊 Holdout pool size: {len(holdout_pool):,} (excluded {len(all_discovery_indices)} discovery samples)")

# Stratified sampling from holdout pool: 350 phone, 150 email
phone_holdout = holdout_pool[holdout_pool['contact_channel'].str.lower().isin(['phone', 'voice'])].copy()
email_holdout = holdout_pool[holdout_pool['contact_channel'].str.lower() == 'email'].copy()

# Sample from each channel
phone_eval_size = min(700, len(phone_holdout))
email_eval_size = min(300, len(email_holdout))

phone_eval = phone_holdout.sample(n=phone_eval_size, random_state=99) if len(phone_holdout) > 0 else pd.DataFrame()
email_eval = email_holdout.sample(n=email_eval_size, random_state=99) if len(email_holdout) > 0 else pd.DataFrame()

# Combine into evaluation set
df_eval = pd.concat([phone_eval, email_eval], ignore_index=False).copy()

print(f"\n✅ Holdout evaluation set created:")
print(f"   📞 Phone/Voice: {len(phone_eval)} samples")
print(f"   📧 Email: {len(email_eval)} samples")
print(f"   📊 Total evaluation set: {len(df_eval)} samples")
print(f"\n🔒 Verification:")
print(f"   Overlap with L1 discovery: {len(set(df_eval.index) & set(l1_used_indices))}")
print(f"   Overlap with L2 discovery: {len(set(df_eval.index) & set(l2_used_indices))}")
print(f"   ✓ All overlaps should be 0!")

# Show sample
print(f"\n📋 Sample from evaluation set:")
for idx, row in df_eval.head(5).iterrows():
    print(f"   {idx}. [{row['contact_channel']}] {row['summary'][:80]}...")


🎯 Creating holdout evaluation set...
📊 Holdout pool size: 30,074 (excluded 6768 discovery samples)

✅ Holdout evaluation set created:
   📞 Phone/Voice: 700 samples
   📧 Email: 300 samples
   📊 Total evaluation set: 1000 samples

🔒 Verification:
   Overlap with L1 discovery: 0
   Overlap with L2 discovery: 0
   ✓ All overlaps should be 0!

📋 Sample from evaluation set:
   29395. [phone] Retailer inquired about the delivery status of a customer's order showing as can...
   5321. [phone] Retailer reported concerns about an Instacart shopper causing disturbances and r...
   10515. [phone] Retailer reported a driver took an order without proper labeling and requested t...
   10723. [phone] Retailer requested cancellation of a delayed customer order due to the customer'...
   32653. [phone] Retailer reported a systematic error preventing order completion and requested a...


In [87]:
# ============================================
# CLEAN UP OLD CHECKPOINTS (Run once)
# ============================================

import os

# Delete old checkpoint files to ensure fresh eval classification
checkpoint_files = [
    'classification_l1_checkpoint.csv',
    'classification_l2_checkpoint.csv',
    'classification_l1_eval_checkpoint.csv',
    'classification_l2_eval_checkpoint.csv'
]

for file in checkpoint_files:
    if os.path.exists(file):
        os.remove(file)
        print(f"🗑️  Deleted: {file}")
    else:
        print(f"✓ Not found: {file}")

print("\n✅ Checkpoint cleanup complete!")

✓ Not found: classification_l1_checkpoint.csv
🗑️  Deleted: classification_l2_checkpoint.csv
✓ Not found: classification_l1_eval_checkpoint.csv
✓ Not found: classification_l2_eval_checkpoint.csv

✅ Checkpoint cleanup complete!


In [88]:
# ============================================
# STEP 4: CLASSIFY Holdout evaluation SUMMARIES INTO L1 CATEGORIES
# ============================================

def classify_to_l1(summary, taxonomy):
    """
    Classify a single summary into an L1 category using GPT
    """
    # Build category list for prompt
    category_list = []
    for l1 in taxonomy['L1_categories']:
        category_list.append(f"{l1['id']}: {l1['name']} - {l1['description']}")
    
    categories_text = "\n".join(category_list)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.1,  # Low temperature for consistent classification
        max_tokens=50,
        messages=[
            {
                'role': 'system',
                'content': """You are a contact reason classifier. Given a summary, pick the most appropriate L1 category.
                
Rules:
- Output ONLY the category ID (e.g., "L1_01")
- Be consistent and precise
- If genuinely uncertain, pick the closest match"""
            },
            {
                'role': 'user',
                'content': f"""Categories:
{categories_text}

Summary: {summary}

Which L1 category best fits this summary? Output only the ID (e.g., "L1_01"):"""
            }
        ]
    )
    
    return response.choices[0].message.content.strip()


def classify_l1_parallel(df, taxonomy, max_workers=10, checkpoint_file='classification_l1_checkpoint.csv'):
    """
    Classify all summaries into L1 categories in parallel
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    import time
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['L1_category'].notna().sum()
        print(f"✅ Already classified: {already_done} rows")
    else:
        df = df.copy()
        df['L1_category'] = None
        df['L1_confidence'] = None
    
    # Get rows that still need classification
    to_process = df[df['L1_category'].isna()]
    
    if len(to_process) == 0:
        print("🎉 All rows already classified!")
        return df
    
    print(f"🚀 Classifying {len(to_process):,} summaries into L1 categories...")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / max_workers / 60:.1f} minutes\n")
    
    def process_row(idx, row):
        try:
            l1_cat = classify_to_l1(row['summary'], taxonomy)
            return idx, l1_cat, None
        except Exception as e:
            return idx, None, str(e)
    
    completed_count = 0
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="L1 Classification"):
            idx, l1_cat, error = future.result()
            
            if l1_cat:
                df.at[idx, 'L1_category'] = l1_cat
            if error and completed_count < 5:
                print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint every 500 rows
            if completed_count % 500 == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%), ETA: {eta_seconds/60:.1f}min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\n✅ L1 Classification complete! ({elapsed/60:.1f} minutes)")
    
    return df


# Run L1 classification ON EVALUATION SET ONLY
print("🎯 Starting L1 classification for EVALUATION SET ONLY...")
print(f"   Classifying {len(df_eval)} holdout samples (not used in discovery)\n")
df_with_l1 = classify_l1_parallel(df_eval, l2_complete_taxonomy, max_workers=10)

# Show distribution
print("\n" + "="*80)
print("📊 L1 CATEGORY DISTRIBUTION")
print("="*80)
l1_dist = df_with_l1['L1_category'].value_counts()
for cat_id, count in l1_dist.items():
    # Find category name
    cat_name = next((c['name'] for c in l2_complete_taxonomy['L1_categories'] if c['id'] == cat_id), cat_id)
    print(f"{cat_id}: {cat_name:30s} - {count:6,} ({count/len(df_with_l1)*100:5.1f}%)")


🎯 Starting L1 classification for EVALUATION SET ONLY...
   Classifying 1000 holdout samples (not used in discovery)

🚀 Classifying 1,000 summaries into L1 categories...
⏱️  Estimated time: 0.8 - 1.7 minutes



L1 Classification:  51%|█████     | 508/1000 [00:20<00:19, 25.40it/s]


💾 Checkpoint: 500/1000 (50.0%), ETA: 0.3min


L1 Classification: 100%|██████████| 1000/1000 [00:43<00:00, 22.77it/s]


💾 Checkpoint: 1000/1000 (100.0%), ETA: 0.0min

✅ L1 Classification complete! (0.7 minutes)

📊 L1 CATEGORY DISTRIBUTION
L1_01: Order Status and Delivery Issues -    280 ( 28.0%)
L1_02: Order Cancellations and Modifications -    280 ( 28.0%)
L1_04: Technical and App Issues       -    187 ( 18.7%)
L1_03: Payment and Refund Issues      -    141 ( 14.1%)
L1_05: Shopper and Driver Issues      -     94 (  9.4%)
L1_06: Account and Membership Issues  -     18 (  1.8%)


In [89]:
# ============================================
# STEP 5: CLASSIFY ALL SUMMARIES INTO L2 CATEGORIES
# ============================================

def classify_to_l2(summary, l1_category, taxonomy):
    """
    Classify a summary into an L2 category given its L1
    """
    # Find the L1 category details
    l1_cat = next((c for c in taxonomy['L1_categories'] if c['id'] == l1_category), None)
    
    if not l1_cat or 'L2_categories' not in l1_cat:
        return None
    
    # Build L2 category list for this L1
    l2_list = []
    for l2 in l1_cat['L2_categories']:
        l2_list.append(f"{l2['id']}: {l2['name']} - {l2['description']}")
    
    l2_text = "\n".join(l2_list)
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-11-20",
        temperature=0.1,
        max_tokens=50,
        messages=[
            {
                'role': 'system',
                'content': """You are a contact reason classifier. Given a summary and its L1 category, pick the most appropriate L2 subcategory.

Rules:
- Output ONLY the L2 category ID (e.g., "L1_01_L2_03")
- Be consistent and precise"""
            },
            {
                'role': 'user',
                'content': f"""L1 Category: {l1_cat['name']}

L2 Subcategories:
{l2_text}

Summary: {summary}

Which L2 subcategory best fits? Output only the ID:"""
            }
        ]
    )
    
    return response.choices[0].message.content.strip()


def classify_l2_parallel(df, taxonomy, max_workers=10, checkpoint_file='classification_l2_checkpoint.csv'):
    """
    Classify all summaries into L2 categories based on their L1
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm import tqdm
    import time
    
    # Check if checkpoint exists
    if os.path.exists(checkpoint_file):
        print(f"📁 Loading checkpoint from {checkpoint_file}")
        df = pd.read_csv(checkpoint_file)
        already_done = df['L2_category'].notna().sum()
        print(f"✅ Already classified: {already_done} rows")
    else:
        df = df.copy()
        df['L2_category'] = None
    
    # Get rows that still need L2 classification (must have L1 first)
    to_process = df[(df['L1_category'].notna()) & (df['L2_category'].isna())]
    
    if len(to_process) == 0:
        print("🎉 All rows already classified!")
        return df
    
    print(f"🚀 Classifying {len(to_process):,} summaries into L2 subcategories...")
    print(f"⏱️  Estimated time: {len(to_process) / (max_workers * 2) / 60:.1f} - {len(to_process) / max_workers / 60:.1f} minutes\n")
    
    def process_row(idx, row):
        try:
            l2_cat = classify_to_l2(row['summary'], row['L1_category'], taxonomy)
            return idx, l2_cat, None
        except Exception as e:
            return idx, None, str(e)
    
    completed_count = 0
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_idx = {
            executor.submit(process_row, idx, row): idx 
            for idx, row in to_process.iterrows()
        }
        
        for future in tqdm(as_completed(future_to_idx), total=len(future_to_idx), desc="L2 Classification"):
            idx, l2_cat, error = future.result()
            
            if l2_cat:
                df.at[idx, 'L2_category'] = l2_cat
            if error and completed_count < 5:
                print(f"\n⚠️  Error on row {idx}: {error}")
            
            completed_count += 1
            
            # Checkpoint every 500 rows
            if completed_count % 500 == 0:
                df.to_csv(checkpoint_file, index=False)
                elapsed = time.time() - start_time
                rate = completed_count / elapsed
                remaining = len(to_process) - completed_count
                eta_seconds = remaining / rate if rate > 0 else 0
                print(f"\n💾 Checkpoint: {completed_count}/{len(to_process)} ({completed_count/len(to_process)*100:.1f}%), ETA: {eta_seconds/60:.1f}min")
    
    # Final save
    df.to_csv(checkpoint_file, index=False)
    
    elapsed = time.time() - start_time
    print(f"\n✅ L2 Classification complete! ({elapsed/60:.1f} minutes)")
    
    return df


# Run L2 classification
print("🎯 Starting L2 classification for all summaries...")
df_with_l2 = classify_l2_parallel(df_with_l1, l2_complete_taxonomy, max_workers=10)

# Show L2 distribution within each L1
print("\n" + "="*80)
print("📊 L2 CATEGORY DISTRIBUTION (within each L1)")
print("="*80)

for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_with_l2[df_with_l2['L1_category'] == l1_cat['id']]
    
    if len(l1_data) == 0:
        continue
    
    print(f"\n{l1_cat['id']}: {l1_cat['name']} (Total: {len(l1_data):,})")
    l2_dist = l1_data['L2_category'].value_counts()
    
    for l2_id, count in l2_dist.items():
        # Find L2 name
        l2_name = next((l2['name'] for l2 in l1_cat.get('L2_categories', []) if l2['id'] == l2_id), l2_id)
        print(f"   └─ {l2_id}: {l2_name:35s} - {count:5,} ({count/len(l1_data)*100:5.1f}%)")


🎯 Starting L2 classification for all summaries...
🚀 Classifying 1,000 summaries into L2 subcategories...
⏱️  Estimated time: 0.8 - 1.7 minutes



L2 Classification:  50%|█████     | 503/1000 [00:24<00:26, 18.95it/s]


💾 Checkpoint: 500/1000 (50.0%), ETA: 0.4min


L2 Classification: 100%|██████████| 1000/1000 [00:48<00:00, 20.73it/s]


💾 Checkpoint: 1000/1000 (100.0%), ETA: 0.0min

✅ L2 Classification complete! (0.8 minutes)

📊 L2 CATEGORY DISTRIBUTION (within each L1)

L1_01: Order Status and Delivery Issues (Total: 280)
   └─ L1_01_L2_01: Order Status Updates                -   104 ( 37.1%)
   └─ L1_01_L2_03: Missing Deliveries                  -    64 ( 22.9%)
   └─ L1_01_L2_02: Delayed Deliveries                  -    42 ( 15.0%)
   └─ L1_01_L2_07: Incorrect or Missing Items          -    26 (  9.3%)
   └─ L1_01_L2_04: Incorrect Delivery Location         -    17 (  6.1%)
   └─ L1_01_L2_06: Delivery Proof and Verification     -    16 (  5.7%)
   └─ L1_01_L2_08: Address Updates and Delivery Instructions -    10 (  3.6%)
   └─ L1_01_L2_05: Order Cancellation Requests         -     1 (  0.4%)

L1_02: Order Cancellations and Modifications (Total: 280)
   └─ L1_02_L2_02: Retailer-Initiated Cancellations    -   223 ( 79.6%)
   └─ L1_02_L2_08: Order Modification Requests         -    18 (  6.4%)
   └─ L1_02_L2_01: Custo

In [90]:
# ============================================
# STEP 6: VALIDATION & QUALITY REVIEW
# ============================================

print("🔍 TAXONOMY QUALITY REVIEW")
print("="*80)

# 1. Check classification completeness
total_records = len(df_with_l2)
l1_classified = df_with_l2['L1_category'].notna().sum()
l2_classified = df_with_l2['L2_category'].notna().sum()

print(f"\n📊 Classification Completeness:")
print(f"   Total records: {total_records:,}")
print(f"   L1 classified: {l1_classified:,} ({l1_classified/total_records*100:.1f}%)")
print(f"   L2 classified: {l2_classified:,} ({l2_classified/total_records*100:.1f}%)")

# 2. Check for category balance
print(f"\n⚖️  Category Balance Check:")
l1_dist = df_with_l2['L1_category'].value_counts()
for cat_id, count in l1_dist.items():
    pct = count / total_records * 100
    cat_name = next((c['name'] for c in l2_complete_taxonomy['L1_categories'] if c['id'] == cat_id), cat_id)
    status = "✅" if 5 <= pct <= 40 else "⚠️ "
    print(f"   {status} {cat_name:30s}: {pct:5.1f}%")

# 3. Sample validation - show random examples from each L1
print(f"\n📋 Sample Classifications (5 random per L1):")
for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_with_l2[df_with_l2['L1_category'] == l1_cat['id']]
    
    if len(l1_data) == 0:
        continue
    
    print(f"\n{l1_cat['name']}:")
    samples = l1_data.sample(min(5, len(l1_data)))
    
    for idx, row in samples.iterrows():
        l2_name = "N/A"
        if pd.notna(row.get('L2_category')):
            l2 = next((l2 for l2 in l1_cat.get('L2_categories', []) if l2['id'] == row['L2_category']), None)
            if l2:
                l2_name = l2['name']
        
        print(f"   └─ L2: {l2_name}")
        print(f"      Summary: {row['summary'][:100]}...")

# 4. Add category names to dataframe for export
def add_category_names(df, taxonomy):
    """Add human-readable category names"""
    df = df.copy()
    
    # Add L1 names
    l1_map = {c['id']: c['name'] for c in taxonomy['L1_categories']}
    df['L1_category_name'] = df['L1_category'].map(l1_map)
    
    # Add L2 names
    l2_map = {}
    for l1_cat in taxonomy['L1_categories']:
        for l2_cat in l1_cat.get('L2_categories', []):
            l2_map[l2_cat['id']] = l2_cat['name']
    df['L2_category_name'] = df['L2_category'].map(l2_map)
    
    return df

df_final = add_category_names(df_with_l2, l2_complete_taxonomy)

print("\n✅ Validation complete! Ready for export.")


🔍 TAXONOMY QUALITY REVIEW

📊 Classification Completeness:
   Total records: 1,000
   L1 classified: 1,000 (100.0%)
   L2 classified: 1,000 (100.0%)

⚖️  Category Balance Check:
   ✅ Order Status and Delivery Issues:  28.0%
   ✅ Order Cancellations and Modifications:  28.0%
   ✅ Technical and App Issues      :  18.7%
   ✅ Payment and Refund Issues     :  14.1%
   ✅ Shopper and Driver Issues     :   9.4%
   ⚠️  Account and Membership Issues :   1.8%

📋 Sample Classifications (5 random per L1):

Order Status and Delivery Issues:
   └─ L2: Order Status Updates
      Summary: Customer reported an issue with catering delivery order #92535 and was added to support request PSO-...
   └─ L2: Order Status Updates
      Summary: Retailer inquired about the delivery status of a customer's order....
   └─ L2: Incorrect Delivery Location
      Summary: Retailer inquired whether a customer's order was delivered to the correct address....
   └─ L2: Missing Deliveries
      Summary: Customer reported m

In [91]:
# ============================================
# STEP 7: EXPORT FINAL TAXONOMY & CLASSIFIED DATA
# ============================================

print("💾 EXPORTING FINAL RESULTS")
print("="*80)

# 1. Export classified data with all fields
output_file = '/Users/ashleyhan/Documents/data/cx_transcripts_with_taxonomy.csv'
df_final.to_csv(output_file, index=False)
print(f"\n✅ Exported classified data to: {output_file}")
print(f"   Columns: {', '.join(df_final.columns)}")

# 2. Export taxonomy definition
taxonomy_file = '/Users/ashleyhan/Documents/data/cx_taxonomy_final.json'
with open(taxonomy_file, 'w') as f:
    json.dump(l2_complete_taxonomy, f, indent=2)
print(f"\n✅ Exported taxonomy definition to: {taxonomy_file}")

# 3. Create summary report
summary_report = {
    'metadata': {
        'total_records': len(df_final),
        'date_created': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
        'channels': df_final['contact_channel'].value_counts().to_dict()
    },
    'L1_distribution': {},
    'L2_distribution': {}
}

# L1 distribution
for l1_cat in l2_complete_taxonomy['L1_categories']:
    l1_data = df_final[df_final['L1_category'] == l1_cat['id']]
    count = len(l1_data)
    
    summary_report['L1_distribution'][l1_cat['id']] = {
        'name': l1_cat['name'],
        'description': l1_cat['description'],
        'count': count,
        'percentage': round(count / len(df_final) * 100, 2)
    }
    
    # L2 distribution within this L1
    l2_dist = {}
    for l2_cat in l1_cat.get('L2_categories', []):
        l2_count = len(df_final[df_final['L2_category'] == l2_cat['id']])
        if l2_count > 0:
            l2_dist[l2_cat['id']] = {
                'name': l2_cat['name'],
                'count': l2_count,
                'percentage_within_l1': round(l2_count / max(count, 1) * 100, 2)
            }
    
    summary_report['L2_distribution'][l1_cat['id']] = l2_dist

report_file = '/Users/ashleyhan/Documents/data/cx_taxonomy_report.json'
with open(report_file, 'w') as f:
    json.dump(summary_report, f, indent=2)
print(f"\n✅ Exported summary report to: {report_file}")

# 4. Create human-readable summary
print("\n" + "="*80)
print("📊 FINAL TAXONOMY SUMMARY")
print("="*80)
print(f"\nTotal Contacts Classified: {len(df_final):,}")
print(f"Classification Rate: {df_final['L2_category'].notna().sum() / len(df_final) * 100:.1f}%")

print(f"\n🏷️  L1 Categories ({len(l2_complete_taxonomy['L1_categories'])}):")
for l1_id, l1_data in summary_report['L1_distribution'].items():
    print(f"\n   {l1_id}: {l1_data['name']} ({l1_data['percentage']:.1f}%)")
    print(f"   {l1_data['description']}")
    
    # Show L2s
    if l1_id in summary_report['L2_distribution']:
        l2_cats = summary_report['L2_distribution'][l1_id]
        print(f"   L2 Subcategories ({len(l2_cats)}):")
        for l2_id, l2_data in l2_cats.items():
            print(f"      └─ {l2_data['name']}: {l2_data['percentage_within_l1']:.1f}% of {l1_data['name']}")

print("\n" + "="*80)
print("✨ TAXONOMY BUILD COMPLETE!")
print("="*80)
print("\nFiles created:")
print(f"1. {output_file}")
print(f"2. {taxonomy_file}")
print(f"3. {report_file}")
print("\nNext steps:")
print("- Review sample classifications for quality")
print("- Refine categories if needed and re-run classification")
print("- Use taxonomy for reporting, routing, and analysis")


💾 EXPORTING FINAL RESULTS

✅ Exported classified data to: /Users/ashleyhan/Documents/data/cx_transcripts_with_taxonomy.csv
   Columns: primary_contact_id, contact_channel, transcript_created_date_at_utc, is_retail_agent, subject, transcript, summary, error, L1_category, L1_confidence, L2_category, L1_category_name, L2_category_name

✅ Exported taxonomy definition to: /Users/ashleyhan/Documents/data/cx_taxonomy_final.json

✅ Exported summary report to: /Users/ashleyhan/Documents/data/cx_taxonomy_report.json

📊 FINAL TAXONOMY SUMMARY

Total Contacts Classified: 1,000
Classification Rate: 100.0%

🏷️  L1 Categories (6):

   L1_01: Order Status and Delivery Issues (28.0%)
   Covers inquiries and complaints about order status, delays, missing deliveries, or incorrect delivery locations.
   L2 Subcategories (8):
      └─ Order Status Updates: 37.1% of Order Status and Delivery Issues
      └─ Delayed Deliveries: 15.0% of Order Status and Delivery Issues
      └─ Missing Deliveries: 22.9% of O

In [92]:
# Load taxonomy from JSON file
import json

with open('taxonomy_complete_proposal.json', 'r') as f:
    taxonomy = json.load(f)

print(f"✅ Loaded taxonomy with {len(taxonomy['L1_categories'])} L1 categories")

# ============================================
# CREATE SLIDE-READY CONTENT
# ============================================

print("="*80)
print("🎯 CX CONTACT REASON TAXONOMY")
print("="*80)
print(f"\n📊 Overview:")
print(f"   • {len(taxonomy['L1_categories'])} Top-Level (L1) Categories")
print(f"   • {sum(len(l1.get('L2_categories', [])) for l1 in taxonomy['L1_categories'])} Subcategories (L2)")
print(f"   • Built from analysis of 36,842 contact transcripts")
print(f"   • Stratified sampling: 70% phone/voice, 30% email")

print(f"\n📋 L1 Categories:\n")

for i, l1 in enumerate(taxonomy['L1_categories'], 1):
    print(f"{i}. {l1['name']} (~{l1.get('estimated_pct', 0)}%)")
    print(f"   {l1['description']}")
    print(f"   🔍 L2 Subcategories ({len(l1.get('L2_categories', []))}):")
    
    for l2 in l1.get('L2_categories', []):
        print(f"      • {l2['name']}")
    print()

# Create markdown for slides
with open('taxonomy_for_slides.md', 'w') as f:
    f.write("# CX Contact Reason Taxonomy\n\n")
    f.write("## Overview\n\n")
    f.write(f"- **{len(taxonomy['L1_categories'])}** Top-Level Categories\n")
    f.write(f"- **{sum(len(l1.get('L2_categories', [])) for l1 in taxonomy['L1_categories'])}** Subcategories\n")
    f.write(f"- Built from **36,842** contact transcripts\n\n")
    
    f.write("---\n\n")
    
    for l1 in taxonomy['L1_categories']:
        f.write(f"## {l1['name']}\n\n")
        f.write(f"**{l1['description']}**\n\n")
        f.write(f"### Subcategories:\n\n")
        for l2 in l1.get('L2_categories', []):
            f.write(f"- **{l2['name']}**: {l2['description']}\n")
        f.write(f"\n---\n\n")

print("💾 Saved markdown for slides to: taxonomy_for_slides.md")

✅ Loaded taxonomy with 6 L1 categories
🎯 CX CONTACT REASON TAXONOMY

📊 Overview:
   • 6 Top-Level (L1) Categories
   • 48 Subcategories (L2)
   • Built from analysis of 36,842 contact transcripts
   • Stratified sampling: 70% phone/voice, 30% email

📋 L1 Categories:

1. Order Status and Delivery Issues (~30%)
   Covers inquiries and complaints about order status, delays, missing deliveries, or incorrect delivery locations.
   🔍 L2 Subcategories (8):
      • Order Status Updates
      • Delayed Deliveries
      • Missing Deliveries
      • Incorrect Delivery Location
      • Order Cancellation Requests
      • Delivery Proof and Verification
      • Incorrect or Missing Items
      • Address Updates and Delivery Instructions

2. Order Cancellations and Modifications (~25%)
   Includes requests to cancel or modify orders due to customer or retailer needs, such as item unavailability, incorrect details, or timing issues.
   🔍 L2 Subcategories (8):
      • Customer-Initiated Cancellations


In [93]:
iq.upload(df_final, "SANDBOX_DB_PII.ASHLEYHAN.CX_RETAILER_CONTACTS_EVAL", if_exists="replace")

True

## gold dataset thats includes the topic summarization and information extraction 

## Contact Information Extraction System

This section implements a comprehensive system to extract structured information from customer service contacts (voice calls and emails).

### Key Improvements:
1. **Multi-Channel Support**: Handles both voice and email contacts with channel-specific guidance
2. **Context-Aware**: Uses subject line for additional context (especially for emails)
3. **Structured Extraction**: Extracts 7 fields including summary, contact reason, audience, retailer, product, order ID, and JIRA ticket
4. **Production-Ready**: Includes parallel processing, checkpointing, error handling, and progress tracking

### How to Use:
See the cells below for the prompt template, helper functions, and complete processing pipeline.


In [109]:
sql_query = """select * from SANDBOX_DB_PII.ASHLEYHAN.CX_RETAILER_CONTACTS_SUMMARY"""
schema_df = iq.query(sql_query)

In [110]:
# Stratified sampling: 350 phone + 150 email = 500 total
# Filter for phone and email only first
schema_df_filtered = schema_df[schema_df["contact_channel"].isin(["phone", "email"])].copy()

# Sample 350 phone records
phone_sample = schema_df_filtered[schema_df_filtered["contact_channel"] == "phone"].sample(n=350, random_state=42)

# Sample 150 email records  
email_sample = schema_df_filtered[schema_df_filtered["contact_channel"] == "email"].sample(n=150, random_state=42)

# Combine the samples
schema_df = pd.concat([phone_sample, email_sample], ignore_index=True)

# Shuffle to mix phone and email records
schema_df = schema_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Verify the distribution
print(f"Total records: {len(schema_df)}")
print(f"\nChannel distribution:")
print(schema_df["contact_channel"].value_counts())
print(f"\nChannel percentages:")
print(schema_df["contact_channel"].value_counts() / len(schema_df))
schema_df.info()




Total records: 500

Channel distribution:
contact_channel
phone    350
email    150
Name: count, dtype: int64

Channel percentages:
contact_channel
phone    0.7
email    0.3
Name: count, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   primary_contact_id              500 non-null    object        
 1   contact_channel                 500 non-null    object        
 2   transcript_created_date_at_utc  500 non-null    datetime64[ns]
 3   is_retail_agent                 500 non-null    int8          
 4   subject                         149 non-null    object        
 5   transcript                      500 non-null    object        
 6   summary                         500 non-null    object        
 7   error                           0 non-null      object        
dtypes: dateti

In [112]:
SchemaSystemPrompt = """
You are a customer experience analyst specializing in contact reason classification. 

CONTEXT PROVIDED:
- Contact Channel: {CONTACT_CHANNEL} (call or email)
- Subject Line: {SUBJECT} (only for email)
- Transcript: {TRANSCRIPT}

YOUR TASK:
Analyze the contact and extract structured information. Consider the subject line for context, but prioritize the transcript content for classification.

EXTRACTION REQUIREMENTS:

1. summary (REQUIRED)
   - Write a single, concise sentence (≤ 25 words) capturing the PRIMARY reason for contact
   - Start with an action verb: "Retailer reported...", "Shopper requested...", "Customer inquired..."
   - Focus on the CORE ISSUE or REQUEST, not secondary concerns
   - Use neutral, descriptive language
   - Include key details: order/delivery issues, payment problems, product concerns, account issues, etc.
   - DO NOT include:
     * PII (names, emails, phone numbers)
     * Agent names or internal processes
     * Phrases like "the email thread" or "the caller"
     * Pleasantries or conversational filler
     * Resolution details (focus on the initial problem)
     * Multiple issues (choose the primary one)

2. contact_reason (REQUIRED)
   - Return EXACTLY one value from this list (copy exactly as written):
     *      Account and Membership Issues :: Account Linking and Information Updates
     * Account and Membership Issues :: Membership Management
     * Account and Membership Issues :: General Account Inquiries
     * Account and Membership Issues :: Account Access Issues
     * Account and Membership Issues :: Account Closure Requests
     * Account and Membership Issues :: Other
     * Order Cancellations and Modifications :: Retailer-Initiated Cancellations
     * Order Cancellations and Modifications :: Order Modification Requests
     * Order Cancellations and Modifications :: Delivery or Address Issues
     * Order Cancellations and Modifications :: Out-of-Stock or Substitution Issues
     * Order Cancellations and Modifications :: Customer-Initiated Cancellations
     * Order Cancellations and Modifications :: Wrong Order or Delivery Mix-Ups
     * Order Cancellations and Modifications :: Payment or Authorization Failures
     * Order Cancellations and Modifications :: System or Technical Issues
     * Order Cancellations and Modifications :: Other
     * Order Status and Delivery Issues :: Delivery Proof and Verification
     * Order Status and Delivery Issues :: Order Status Updates
     * Order Status and Delivery Issues :: Missing Deliveries
     * Order Status and Delivery Issues :: Delayed Deliveries
     * Order Status and Delivery Issues :: Address Updates and Delivery Instructions
     * Order Status and Delivery Issues :: Incorrect Delivery Location
     * Order Status and Delivery Issues :: Incorrect or Missing Items
     * Order Status and Delivery Issues :: Other
     * Payment and Refund Issues :: Declined Payments
     * Payment and Refund Issues :: Overcharges
     * Payment and Refund Issues :: Incorrect Charges for Promotions or Discounts
     * Payment and Refund Issues :: Refund Delays or Missing Refunds
     * Payment and Refund Issues :: Payment Processing Errors
     * Payment and Refund Issues :: Unauthorized or Fraudulent Charges
     * Payment and Refund Issues :: Other
     * Shopper Issues :: Shopper Misconduct
     * Shopper Issues :: Shopper Reassignment
     * Shopper Issues :: Shopper Performance Issues
     * Shopper Issues :: Other
     * Technical and App Issues :: Login and Access Issues
     * Technical and App Issues :: Order Processing Errors
     * Technical and App Issues :: Connectivity and Network Issues
     * Technical and App Issues :: Device Malfunctions
     * Technical and App Issues :: Order Cancellation and Modification Errors
     * Technical and App Issues :: App Performance Issues
     * Technical and App Issues :: Caper Carts Hardware Issues
     * Technical and App Issues :: Communication and Call Issues
     * Technical and App Issues :: Other
     * Store Closures/Hours :: Store Closures and Adjusted Hours
     * Store Closures/Hours :: Power and Network Outages
     * Store Closures/Hours :: Other
     * Other :: Other
   - If none fit well, return "Other :: Other"

3. contact_audience (REQUIRED)
   - Return EXACTLY one value (copy exactly as written):
     * Shopper
     * Internal Employee
     * Retailer / Retailer Account Manager
     * Customer
   - Determine based on who is contacting support, not who is being discussed

4. retailer (optional, can be null)
   - Return the retailer brand name as it appears in context (e.g., "Kroger", "Publix", "Costco", "Sprouts Farmers Market")
   - If multiple retailers appear, choose the one that is the primary subject of the issue
   - If no retailer is mentioned or identifiable, return null

5. product (optional, can be null)
   - Allowed values: "Storefront Pro", "Storefront", "Connect", "LMD", "IPP", "Caper"
   - Map variants/synonyms:
     * "SFP" / "storefront pro" → "Storefront Pro"
     * "last mile delivery" / "last-mile delivery" → "LMD"
   - Special cases:
     * If about Caper carts or smart cart technology → "Caper"
     * If about Instacart Platform Portal → "IPP"
   - Return comma-separated values if multiple products are mentioned (e.g., "Storefront Pro, Caper")
   - If no product is mentioned, return null

6. order_id (optional, can be null)
   - Extract the order ID if mentioned in the transcript
   - Return as a string of numbers only (e.g., "123456789")
   - If multiple order IDs appear, choose the one that is the primary subject of the issue
   - If no order ID is mentioned, return null

7. jira_ticket (optional, can be null)
   - Extract the JIRA ticket if mentioned (format: "PSO-[number]", e.g., "PSO-12345")
   - If multiple tickets appear, choose the one that is the primary subject of the issue
   - If no JIRA ticket is mentioned, return null

8. notes (optional, can be null)
   - Write any additional notes about the contact that are not covered by the other fields
   - Include any details that are interesting or important
   - Do not include PII, agent names, or internal process details

CHANNEL-SPECIFIC GUIDANCE:
- For VOICE contacts: Focus on the conversational flow; the subject is usually null
- For EMAIL contacts: Use the subject line to understand context, but rely on email body for classification

RETURN FORMAT:
Return ONLY valid JSON with exactly these keys (no extra text, no markdown, no code fences):
{{
  "summary": "",
  "contact_reason": "",
  "contact_audience": "",
  "retailer": null,
  "product": null,
  "order_id": null,
  "jira_ticket": null,
  "notes": null
}}
""".strip()

In [ ]:
# Helper function to format the prompt with contact data
def format_extraction_prompt(transcript, subject="", contact_channel="email"):
    """
    Format the SchemaSystemPrompt with actual contact data.
    
    Parameters:
    - transcript (str): The full transcript/email body
    - subject (str): The subject line (use "" if not applicable, e.g., for voice calls)
    - contact_channel (str): Either "voice" or "email"
    
    Returns:
    - str: Formatted prompt ready for GPT
    """
    # Truncate long transcripts if needed (to manage token limits)
    max_transcript_length = 8000
    if len(transcript) > max_transcript_length:
        transcript = transcript[:max_transcript_length] + "\n\n[Transcript truncated for length...]"
    
    # Format the prompt
    formatted_prompt = SchemaSystemPrompt.format(
        CONTACT_CHANNEL=contact_channel,
        SUBJECT=subject if subject else "(No subject provided)",
        TRANSCRIPT=transcript
    )
    
    return formatted_prompt


# Example usage:
"""
# For email contacts:
prompt = format_extraction_prompt(
    transcript=row['transcript'],
    subject=row['subject'],
    contact_channel='email'
)

# For voice/phone contacts:
prompt = format_extraction_prompt(
    transcript=row['transcript'],
    subject='',  # Usually no subject for voice calls
    contact_channel='voice'
)

# Then call OpenAI:
response = client.chat.completions.create(
    model="gpt-4o-2024-11-20",
    temperature=0.1,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

# Parse the JSON response:
result = json.loads(response.choices[0].message.content)
"""


'\n# For email contacts:\nprompt = format_extraction_prompt(\n    transcript=row[\'transcript\'],\n    subject=row[\'subject\'],\n    contact_channel=\'email\'\n)\n\n# For voice/phone contacts:\nprompt = format_extraction_prompt(\n    transcript=row[\'transcript\'],\n    subject=\'\',  # Usually no subject for voice calls\n    contact_channel=\'voice\'\n)\n\n# Then call OpenAI:\nresponse = client.chat.completions.create(\n    model="gpt-4o-2024-11-20",\n    temperature=0.1,\n    messages=[\n        {"role": "user", "content": prompt}\n    ]\n)\n\n# Parse the JSON response:\nresult = json.loads(response.choices[0].message.content)\n'

In [ ]:
# Complete processing function with parallel execution
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import time

def extract_contact_info(transcript, subject="", contact_channel="email"):
    """
    Extract structured information from a single contact using GPT.
    
    Returns:
    - dict: Extracted fields or error information
    """
    try:
        # Format the prompt
        prompt = format_extraction_prompt(transcript, subject, contact_channel)
        
        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4o-2024-11-20",
            temperature=0.1,
            max_tokens=300,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        # Parse JSON response
        result = json.loads(response.choices[0].message.content)
        
        return result
        
    except json.JSONDecodeError as e:
        return {
            "summary": "ERROR: Invalid JSON response",
            "contact_reason": "ERROR",
            "contact_audience": "ERROR",
            "retailer": None,
            "product": None,
            "order_id": None,
            "jira_ticket": None,
            "error_details": str(e),
            "notes": None
        }
    except Exception as e:
        return {
            "summary": f"ERROR: {str(e)}",
            "contact_reason": "ERROR",
            "contact_audience": "ERROR",
            "retailer": None,
            "product": None,
            "order_id": None,
            "jira_ticket": None,
            "error_details": str(e),
            "notes": None
        }


def process_contacts_parallel(df, max_workers=10, checkpoint_every=100, checkpoint_file='contact_extraction_checkpoint.csv'):
    """
    Process contacts in parallel with checkpointing.
    
    Parameters:
    - df: DataFrame with columns: 'transcript', 'subject' (optional), 'contact_channel'
    - max_workers: Number of parallel threads
    - checkpoint_every: Save checkpoint every N records
    - checkpoint_file: Path to checkpoint file
    
    Returns:
    - DataFrame with extracted fields added
    """
    
    # Check for checkpoint file
    if os.path.exists(checkpoint_file):
        print(f"📂 Found checkpoint file: {checkpoint_file}")
        print("   Loading previous progress...")
        checkpoint_df = pd.read_csv(checkpoint_file)
        print(f"   ✅ Loaded {len(checkpoint_df)} previously processed records")
        return checkpoint_df
    
    # Ensure required columns exist
    if 'subject' not in df.columns:
        df['subject'] = ""
    if 'contact_channel' not in df.columns:
        print("⚠️  Warning: 'contact_channel' column not found. Defaulting to 'email'")
        df['contact_channel'] = 'email'
    
    # Prepare results
    results = []
    
    # Helper function for parallel processing
    def process_row(idx, row):
        try:
            result = extract_contact_info(
                transcript=row['transcript'],
                subject=row.get('subject', ''),
                contact_channel=row.get('contact_channel', 'email')
            )
            return idx, result
        except Exception as e:
            return idx, {
                "summary": f"ERROR: {str(e)}",
                "contact_reason": "ERROR",
                "contact_audience": "ERROR",
                "retailer": None,
                "product": None,
                "order_id": None,
                "jira_ticket": None
            }
    
    # Process in parallel
    print(f"🚀 Starting parallel extraction with {max_workers} workers...")
    print(f"   Processing {len(df)} contacts")
    
    start_time = time.time()
    completed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {executor.submit(process_row, idx, row): idx for idx, row in df.iterrows()}
        
        # Process completed tasks with progress bar
        with tqdm(total=len(df), desc="Extracting") as pbar:
            for future in as_completed(futures):
                idx, result = future.result()
                results.append((idx, result))
                completed += 1
                pbar.update(1)
                
                # Checkpoint
                if completed % checkpoint_every == 0:
                    elapsed = time.time() - start_time
                    rate = completed / elapsed if elapsed > 0 else 0
                    eta = (len(df) - completed) / rate / 60 if rate > 0 else 0
                    
                    # Create checkpoint
                    checkpoint_data = []
                    for result_idx, result_data in results:
                        row_data = df.loc[result_idx].to_dict()
                        row_data.update(result_data)
                        checkpoint_data.append(row_data)
                    
                    checkpoint_df = pd.DataFrame(checkpoint_data)
                    checkpoint_df.to_csv(checkpoint_file, index=False)
                    
                    print(f"\n💾 Checkpoint saved. Progress: {completed}/{len(df)} ({100*completed/len(df):.1f}%)")
                    print(f"   Rate: {rate:.1f} rows/sec, ETA: {eta:.1f} min\n")
    
    # Create final DataFrame
    print("\n🎉 Processing complete! Building final DataFrame...")
    final_data = []
    for idx, result in results:
        row_data = df.loc[idx].to_dict()
        row_data.update(result)
        final_data.append(row_data)
    
    result_df = pd.DataFrame(final_data)
    
    # Save final checkpoint
    result_df.to_csv(checkpoint_file, index=False)
    print(f"✅ Final results saved to: {checkpoint_file}")
    
    return result_df




'\n# Assuming you have a DataFrame \'df_contacts\' with columns:\n# - transcript (text)\n# - subject (text, optional)\n# - contact_channel (\'voice\' or \'email\')\n\ndf_extracted = process_contacts_parallel(\n    schema_df,\n    max_workers=10,\n    checkpoint_every=100,\n    checkpoint_file=\'contact_extraction_results.csv\'\n)\n\n# Check results\nprint(df_extracted[[\'summary\', \'contact_reason\', \'contact_audience\', \'retailer\',\'product\',\'order_id\',\'jira_ticket\',\'notes\']].head())\n\n# Count distribution\nprint("\nContact Reason Distribution:")\nprint(df_extracted[\'contact_reason\'].value_counts())\n\nprint("\nContact Audience Distribution:")\nprint(df_extracted[\'contact_audience\'].value_counts())\n'

In [137]:
# Assuming you have a DataFrame 'df_contacts' with columns:
# - transcript (text)
# - subject (text, optional)
# - contact_channel ('voice' or 'email')

df_extracted = process_contacts_parallel(
    schema_df,
    max_workers=10,
    checkpoint_every=100,
    checkpoint_file='contact_extraction_results.csv'
)

# Check results
print(df_extracted[['summary', 'contact_reason', 'contact_audience', 'retailer','product','order_id','jira_ticket','notes']].head())

# Count distribution
print("\nContact Reason Distribution:")
print(df_extracted['contact_reason'].value_counts())

print("\nContact Audience Distribution:")
print(df_extracted['contact_audience'].value_counts())

📂 Found checkpoint file: contact_extraction_results.csv
   Loading previous progress...
   ✅ Loaded 500 previously processed records
                                             summary  \
0  Retailer reported a customer's inability to co...   
1  Customer requested to block a driver for an or...   
2  Retailer reported a change in store hours for ...   
3  Retailer reported a resolved power outage at S...   
4  Retailer reported a declined payment due to an...   

                                      contact_reason  \
0  Order Cancellations and Modifications :: Order...   
1             Shopper Issues :: Shopper Reassignment   
2  Store Closures/Hours :: Store Closures and Adj...   
3  Store Closures/Hours :: Power and Network Outages   
4     Payment and Refund Issues :: Declined Payments   

                      contact_audience                  retailer product  \
0  Retailer / Retailer Account Manager                       NaN     NaN   
1                             Customer  L

In [141]:
# Define regex patterns and masking functions for PII
import re

# Regex patterns for email and phone
EMAIL_RE = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b')
PHONE_RE = re.compile(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b')

# Masking functions
def mask_email(match):
    """Mask email addresses"""
    email = match.group(0)
    parts = email.split('@')
    if len(parts) == 2:
        return f"[EMAIL-{parts[1]}]"
    return "[EMAIL]"

def mask_phone(match):
    """Mask phone numbers"""
    return "[PHONE]"

def mask_names(text, names=None, spacy_model=None):
    """
    Mask person names in text.
    
    Args:
        text: Input text to mask
        names: Optional list of specific names to mask
        spacy_model: Optional spaCy model for NER-based name detection
    
    Returns:
        Text with names masked as [NAME]
    """
    if not text:
        return text
    
    masked_text = text
    
    # Mask specific names if provided
    if names:
        for name in names:
            masked_text = re.sub(
                rf'\b{re.escape(name)}\b', 
                '[NAME]', 
                masked_text, 
                flags=re.IGNORECASE
            )
    
    # Use spaCy for NER-based name detection if model provided
    if spacy_model:
        try:
            import spacy
            nlp = spacy.load(spacy_model)
            doc = nlp(masked_text)
            # Replace PERSON entities with [NAME]
            for ent in reversed(doc.ents):
                if ent.label_ == 'PERSON':
                    masked_text = masked_text[:ent.start_char] + '[NAME]' + masked_text[ent.end_char:]
        except Exception as e:
            print(f"Warning: Could not use spaCy for name masking: {e}")
    
    return masked_text


def mask_pii_extended(text, mask_names_list=None, spacy_model=None):
    """
    Complete PII masking pipeline that applies all masking functions.
    
    Args:
        text: Input text to mask
        mask_names_list: Optional list of specific names to mask
        spacy_model: Optional spaCy model for NER-based name detection
    
    Returns:
        Text with all PII masked (emails, phones, names)
    """
    if text is None:
        return text
    
    # Convert to string
    masked = str(text)
    
    # Mask emails
    masked = EMAIL_RE.sub(mask_email, masked)
    
    # Mask phone numbers
    masked = PHONE_RE.sub(mask_phone, masked)
    
    # Mask names
    masked = mask_names(masked, names=mask_names_list, spacy_model=spacy_model)
    
    return masked


print("PII masking functions loaded!")
print("Available functions:")
print("  - mask_email(): Mask email addresses")
print("  - mask_phone(): Mask phone numbers")
print("  - mask_names(): Mask person names")
print("  - mask_pii_extended(): Complete PII masking pipeline (recommended)")


PII masking functions loaded!
Available functions:
  - mask_email(): Mask email addresses
  - mask_phone(): Mask phone numbers
  - mask_names(): Mask person names
  - mask_pii_extended(): Complete PII masking pipeline (recommended)


In [142]:
# Apply extended masking to both transcript and transcript_cleaned
custom_names = []  # optionally add known names to mask
spacy_model = None  # optionally set to a spaCy model name

df_extracted = df_extracted.copy()

print("Applying PII masking to transcript columns...")

# Mask original transcript column
if 'transcript' in df_extracted.columns:
    print("  - Masking 'transcript' → 'transcript_masked'")
    df_extracted['transcript_masked'] = df_extracted['transcript'].astype(str).apply(
        lambda t: mask_pii_extended(t, mask_names_list=custom_names, spacy_model=spacy_model)
    )
    print(f"    ✓ Completed {len(df_extracted)} records")

# Mask cleaned transcript column (if it exists)
if 'transcript_cleaned' in df_extracted.columns:
    print("  - Masking 'transcript_cleaned' → 'transcript_cleaned_masked'")
    df_extracted['transcript_cleaned_masked'] = df_extracted['transcript_cleaned'].astype(str).apply(
        lambda t: mask_pii_extended(t, mask_names_list=custom_names, spacy_model=spacy_model)
    )
    print(f"    ✓ Completed {len(df_extracted)} records")
else:
    print("  - 'transcript_cleaned' column not found, skipping")

print(f"\n✅ PII masking complete!")
print(f"Columns in df_extracted: {list(df_extracted.columns)}")

Applying PII masking to transcript columns...
  - Masking 'transcript' → 'transcript_masked'
    ✓ Completed 500 records
  - 'transcript_cleaned' column not found, skipping

✅ PII masking complete!
Columns in df_extracted: ['primary_contact_id', 'contact_channel', 'transcript_created_date_at_utc', 'is_retail_agent', 'subject', 'transcript', 'summary', 'error', 'contact_reason', 'contact_audience', 'retailer', 'product', 'order_id', 'jira_ticket', 'notes', 'transcript_masked', 'subject_masked']


In [145]:
import pandas as pd
import re


# Find one email and one phone example
email_example = df_extracted[df_extracted['contact_channel'] == 'email'].iloc[0] if len(df_extracted[df_extracted['contact_channel'] == 'email']) > 0 else None
phone_example = df_extracted[df_extracted['contact_channel'] == 'phone'].iloc[0] if len(df_extracted[df_extracted['contact_channel'] == 'phone']) > 0 else None

# Enhanced PII masking function
def mask_pii(text):
    if pd.isna(text) or text == "":
        return text
    
    masked_text = str(text)
    
    # 1. SPELLED-OUT PHONE NUMBERS (for voice transcripts)
    number_words = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'oh': '0'
    }
    
    word_pattern = r'\b(?:' + '|'.join(number_words.keys()) + r')\b'
    matches = list(re.finditer(word_pattern, masked_text, re.IGNORECASE))
    
    i = 0
    while i < len(matches):
        sequence_start = i
        sequence_end = i
        
        while sequence_end < len(matches) - 1:
            current_end = matches[sequence_end].end()
            next_start = matches[sequence_end + 1].start()
            if next_start - current_end < 50:
                sequence_end += 1
            else:
                break
        
        sequence_length = sequence_end - sequence_start + 1
        
        if sequence_length >= 7:
            start_pos = matches[sequence_start].start()
            end_pos = matches[sequence_end].end()
            masked_text = masked_text[:start_pos] + '[PHONE_SPOKEN]' + masked_text[end_pos:]
            matches = list(re.finditer(word_pattern, masked_text, re.IGNORECASE))
            i = 0
        else:
            i = sequence_end + 1
    
    # 2. Names in conversational context
    masked_text = re.sub(
        r'(?:my name is|i am|i\'m|this is|named)\s+([a-z]+)',
        r'\1 [NAME]',
        masked_text,
        flags=re.IGNORECASE
    )
    
    # 3. Email addresses
    masked_text = re.sub(
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        '[EMAIL]',
        masked_text
    )
    
    # 4. Phone numbers (numeric)
    masked_text = re.sub(
        r'\b(?:\+?1[-.]?)?\(?([0-9]{3})\)?[-.]?([0-9]{3})[-.]?([0-9]{4})\b',
        '[PHONE]',
        masked_text
    )
    
    # 5. Credit cards
    masked_text = re.sub(
        r'\b(?:\d[ -]*?){13,19}\b',
        '[CREDIT_CARD]',
        masked_text
    )
    
    # 6. Order IDs
    masked_text = re.sub(
        r'\b(?:order|account|id|number|#)[:\s]*([0-9]{6,})\b',
        r'[ORDER_ID]',
        masked_text,
        flags=re.IGNORECASE
    )
    
    # 7. URLs
    masked_text = re.sub(
        r'https?://[^\s]+',
        '[URL]',
        masked_text
    )
    
    return masked_text

# Print email example
if email_example is not None:
    print("="*80)
    print("📧 EMAIL EXAMPLE")
    print("="*80)
    print("\n--- ORIGINAL SUBJECT ---")
    print(email_example['subject'][:200] if pd.notna(email_example['subject']) else "(No subject)")
    print("\n--- MASKED SUBJECT ---")
    print(mask_pii(email_example['subject'])[:200] if pd.notna(email_example['subject']) else "(No subject)")
    print("\n--- ORIGINAL TRANSCRIPT (first 500 chars) ---")
    print(email_example['transcript'][:500])
    print("\n--- MASKED TRANSCRIPT (first 500 chars) ---")
    print(mask_pii(email_example['transcript'])[:500])
    print("\n")

# Print phone example
if phone_example is not None:
    print("="*80)
    print("📱 PHONE/VOICE EXAMPLE")
    print("="*80)
    print("\n--- ORIGINAL SUBJECT ---")
    print(phone_example['subject'][:200] if pd.notna(phone_example['subject']) else "(No subject)")
    print("\n--- MASKED SUBJECT ---")
    print(mask_pii(phone_example['subject'])[:200] if pd.notna(phone_example['subject']) else "(No subject)")
    print("\n--- ORIGINAL TRANSCRIPT (first 600 chars) ---")
    print(phone_example['transcript'][:600])
    print("\n--- MASKED TRANSCRIPT (first 600 chars) ---")
    print(mask_pii(phone_example['transcript'])[:600])
    print("\n")


📧 EMAIL EXAMPLE

--- ORIGINAL SUBJECT ---
[Request received] - Block Driver Request 531900014032044

--- MASKED SUBJECT ---
[Request received] - Block Driver Request [CREDIT_CARD]

--- ORIGINAL TRANSCRIPT (first 500 chars) ---
Message 1 (2025-09-28 17:28:41.000 Z):
Your request (945768) has been received and is being reviewed by our support staff.

To add additional comments, reply to this email.

*******

Votre demande (945768) a été reçue et est en cours d'examen par notre équipe d'assistance.

Pour ajouter des commentaires supplémentaires, répondez à cet e-mail.

--------------------------------
This email is a service from Loblaw Companies Limited.









[02EMRE-L5N0Z]

To unsubscribe from this group and stop r

--- MASKED TRANSCRIPT (first 500 chars) ---
Message 1 (2025-09-28 17:28:41.000 Z):
Your request (945768) has been received and is being reviewed by our support staff.

To add additional comments, reply to this email.

*******

Votre demande (945768) a été reçue et est en

In [147]:
iq.upload(df_extracted, "SANDBOX_DB_PII.ASHLEYHAN.CX_RETAILER_CONTACTS_SCHEMA", if_exists="replace")

True

In [146]:
df_extracted.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   primary_contact_id              500 non-null    object 
 1   contact_channel                 500 non-null    object 
 2   transcript_created_date_at_utc  500 non-null    object 
 3   is_retail_agent                 500 non-null    int64  
 4   subject                         149 non-null    object 
 5   transcript                      500 non-null    object 
 6   summary                         500 non-null    object 
 7   error                           0 non-null      float64
 8   contact_reason                  500 non-null    object 
 9   contact_audience                500 non-null    object 
 10  retailer                        308 non-null    object 
 11  product                         62 non-null     object 
 12  order_id                        149 